In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import time
import matplotlib.pyplot as plt

In [ ]:
Linear Voted Perceptron (d=1)

In [ ]:
def train_linear_voted_perceptron(X_tr, y_bin, epochs):
    """Trains a Linear Voted Perceptron (d=1)."""
    V = [np.zeros(X_tr.shape[1])]
    C = [0]

    for epoch in range(epochs):
        for i in range(len(X_tr)):
            x_i = X_tr[i]
            y_i = y_bin[i]

            y_hat = 1 if np.dot(V[-1], x_i) >= 0 else -1

            if y_hat == y_i:
                # If correct, increment the survival count of the current weight vector
                C[-1] += 1
            else:
                # If wrong, append a new weight vector and initialize its count
                V.append(V[-1] + y_i * x_i)
                C.append(1)

    return np.array(V), np.array(C)

In [ ]:
def predict_linear_voted_perceptron(X_test, V_mat, C_vec):
    """Predicts using the trained Linear Voted Perceptron."""
    # Calculate dot products of all test samples with all historical weight vectors
    dots = np.dot(X_test, V_mat.T)
    
    # Get the sign (prediction) for every single weight vector
    signs = np.where(dots >= 0, 1, -1)
    
    # Compute the voted score by taking the dot product of signs and survival counts (C)
    score_vote = np.dot(signs, C_vec)
    
    # Final prediction is the sign of the accumulated votes
    return np.where(score_vote >= 0, 1, -1)

In [ ]:
Polynomial Kernel Voted Perceptron (d>1)

In [ ]:
def train_kernel_voted_perceptron(X_tr, y_bin, epochs, d):
    """Trains a Polynomial Kernel Voted Perceptron (d > 1)."""
    mistakes = []
    C = [0]

    for epoch in range(epochs):
        for i in range(len(X_tr)):
            x_i = X_tr[i]
            y_i = y_bin[i]

            if len(mistakes) == 0:
                v_dot_x = 0
            else:
                # Vectorized kernel computation against all historical mistakes
                M_x = np.array([m[0] for m in mistakes])
                M_y = np.array([m[1] for m in mistakes])
                v_dot_x = np.sum(M_y * ((np.dot(M_x, x_i) + 1.0) ** d))

            y_hat = 1 if v_dot_x >= 0 else -1

            if y_hat == y_i:
                # If correct, increment survival count
                C[-1] += 1
            else:
                # If wrong, record the mistake and initialize count for the new state
                mistakes.append((x_i, y_i))
                C.append(1)

    return mistakes, np.array(C)

In [ ]:
def predict_kernel_voted_perceptron(X_test, mistakes, C_vec, d):
    """Predicts using the trained Polynomial Kernel Voted Perceptron."""
    if len(mistakes) == 0:
        return np.ones(len(X_test))
        
    M_x = np.array([m[0] for m in mistakes])
    M_y = np.array([m[1] for m in mistakes])

    # Compute the polynomial kernel matrix against all historical mistakes
    K_matrix = (np.dot(X_test, M_x.T) + 1.0) ** d
    K_weighted = K_matrix * M_y

    # Calculate the cumulative sum of kernel evaluations to recreate the state at each step
    H_vals = np.hstack([np.zeros((len(X_test), 1)), np.cumsum(K_weighted, axis=1)])
    
    # Get the sign (prediction) at each historical state
    signs = np.where(H_vals >= 0, 1, -1)
    
    # Compute the voted score by taking the dot product of signs and survival counts (C)
    score_vote = np.dot(signs, C_vec)
    
    # Final prediction is the sign of the accumulated votes
    return np.where(score_vote >= 0, 1, -1)